# Stage 2 – Data Acquisition: Amazon India Air Conditioner Scraper

**Project:** Web Scraping & Product Price Analysis (Air Conditioners, Amazon India)

**Steps in this notebook**
1. Install libraries
2. Import libraries
3. Configuration
4. Helper functions (headers, cleaning, fetching)
5. Parse one product card
6. Scrape all pages
7. Remove duplicates and save CSV
8. Preview and quick check

> **Note:** Amazon blocks bots. If you get a CAPTCHA or HTTP 503, run this notebook on your own laptop (Jupyter / VS Code) instead of Colab, increase `DELAY_RANGE`, or try again later. Use only for educational purposes and keep requests slow.

## Step 1: Install the libraries
`requests` downloads pages, `beautifulsoup4` + `lxml` parse HTML, `pandas` stores the data.

In [ ]:
!pip install requests beautifulsoup4 pandas lxml

## Step 2: Import the libraries

In [ ]:
import random
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

print("Libraries imported successfully.")

## Step 3: Configuration
Search keyword, number of pages, delay between requests and output file name.

In [ ]:
BASE_URL = "https://www.amazon.in"
SEARCH_URL = BASE_URL + "/s"
SEARCH_QUERY = "air conditioner"
MAX_PAGES = 20                 # about 20-25 pages gives roughly 700 listings
DELAY_RANGE = (3, 7)           # random wait (seconds) between pages
MAX_RETRIES = 3
OUTPUT_FILE = "Air_Conditioners_Raw.csv"

MAIN_CATEGORY = "appliances"
SUB_CATEGORY = "Air Conditioners"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

## Step 4: Helper functions
- `get_headers()` makes the request look like a normal browser.
- `clean_price`, `clean_rating`, `clean_count` convert text such as `₹32,999` or `4.2 out of 5 stars` into numbers.
- `fetch_page()` downloads one results page with retries and CAPTCHA detection.

In [ ]:
def get_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "en-IN,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": BASE_URL + "/",
        "Connection": "keep-alive",
    }


def clean_price(text):
    """'₹32,999' -> 32999.0"""
    if not text:
        return None
    digits = re.sub(r"[^\d.]", "", text.replace(",", ""))
    try:
        return float(digits) if digits else None
    except ValueError:
        return None


def clean_rating(text):
    """'4.2 out of 5 stars' -> 4.2"""
    if not text:
        return None
    m = re.search(r"(\d+(\.\d+)?)", text)
    return float(m.group(1)) if m else None


def clean_count(text):
    """'2,255' -> 2255.0 ; '1.2K' -> 1200.0"""
    if not text:
        return None
    t = text.strip().replace(",", "").replace("(", "").replace(")", "")
    m = re.search(r"(\d+(\.\d+)?)\s*([KkMm]?)", t)
    if not m:
        return None
    value = float(m.group(1))
    suffix = m.group(3).lower()
    if suffix == "k":
        value *= 1_000
    elif suffix == "m":
        value *= 1_000_000
    return value


def fetch_page(session, page):
    """Download one search-result page with retries. Returns HTML or None."""
    params = {"k": SEARCH_QUERY, "page": page}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(SEARCH_URL, params=params, headers=get_headers(), timeout=20)
            if resp.status_code == 200:
                if "captcha" in resp.text.lower() and "s-search-result" not in resp.text:
                    print(f"  [!] CAPTCHA on page {page} (attempt {attempt}). Waiting...")
                    time.sleep(random.uniform(15, 30))
                    continue
                return resp.text
            print(f"  [!] HTTP {resp.status_code} on page {page} (attempt {attempt})")
        except requests.RequestException as e:
            print(f"  [!] Request error on page {page}: {e}")
        time.sleep(random.uniform(5, 10))
    return None

print("Helper functions ready.")

## Step 5: Parse one product card
Each product on the page is a `div` with `data-component-type='s-search-result'`. We extract name, link, image, rating, number of ratings, selling price and MRP using CSS selectors. If a field is empty later, inspect the page and update these selectors.

In [ ]:
def parse_card(card):
    asin = card.get("data-asin", "").strip()
    if not asin:
        return None

    h2 = card.find("h2")
    name = h2.get_text(" ", strip=True) if h2 else None
    link_tag = (h2.find("a") if h2 else None) or card.find("a", href=True)
    link = urljoin(BASE_URL, link_tag["href"]) if link_tag and link_tag.get("href") else None

    img = card.find("img", class_="s-image")
    image = img["src"] if img and img.get("src") else None

    rating_tag = card.select_one("span.a-icon-alt")
    ratings = clean_rating(rating_tag.get_text(strip=True) if rating_tag else None)

    count_tag = (card.select_one("span.a-size-base.s-underline-text")
                 or card.select_one("a[href*='customerReviews'] span"))
    no_of_ratings = clean_count(count_tag.get_text(strip=True) if count_tag else None)

    price_tag = card.select_one("span.a-price:not(.a-text-price) span.a-offscreen")
    discount_price = clean_price(price_tag.get_text(strip=True) if price_tag else None)

    mrp_tag = card.select_one("span.a-price.a-text-price[data-a-strike='true'] span.a-offscreen")
    actual_price = clean_price(mrp_tag.get_text(strip=True) if mrp_tag else None)

    if not name:
        return None

    return {
        "asin": asin,
        "name": name,
        "main_category": MAIN_CATEGORY,
        "sub_category": SUB_CATEGORY,
        "image": image,
        "link": link,
        "ratings": ratings,
        "no_of_ratings": no_of_ratings,
        "discount_price": discount_price,
        "actual_price": actual_price,
    }


def parse_page(html):
    soup = BeautifulSoup(html, "lxml")
    cards = soup.select("div[data-component-type='s-search-result']")
    products = []
    for card in cards:
        item = parse_card(card)
        if item:
            products.append(item)
    return products

print("Parser ready.")

## Step 6: Scrape all pages
Loops through the result pages, waits a random time between requests, and saves progress after every page so nothing is lost if Amazon blocks us.

In [ ]:
session = requests.Session()
all_products = []

for page in range(1, MAX_PAGES + 1):
    print(f"Scraping page {page}/{MAX_PAGES} ...")
    html = fetch_page(session, page)
    if html is None:
        print("  Could not fetch this page. Stopping early.")
        break

    items = parse_page(html)
    print(f"  Found {len(items)} listings.")
    if not items:
        print("  No listings found (end of results or blocked). Stopping.")
        break

    all_products.extend(items)
    pd.DataFrame(all_products).to_csv(OUTPUT_FILE, index=False)   # save progress
    time.sleep(random.uniform(*DELAY_RANGE))                      # be polite

print("\nTotal listings collected (with duplicates):", len(all_products))

## Step 7: Remove duplicates and save the CSV
Sponsored ads repeat the same product, so we drop duplicates using the ASIN (Amazon's unique product ID), then keep the 9 raw columns used in the project.

In [ ]:
df = pd.DataFrame(all_products)

before = len(df)
df = df.drop_duplicates(subset="asin").drop(columns="asin").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate listings.")

df = df[["name", "main_category", "sub_category", "image", "link",
         "ratings", "no_of_ratings", "discount_price", "actual_price"]]

df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} rows x {df.shape[1]} columns to '{OUTPUT_FILE}'.")

## Step 8: Preview and quick check
Confirm the shape, data types and missing values of the scraped data.

In [ ]:
print("Shape:", df.shape)
print()
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())
df.head()

In [ ]:
# (Optional) Download the CSV if you are running in Google Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("Not running in Colab - the CSV is saved in the current folder.")